# Profundidades de eclipse v2

Vuelve a medir la profundidad del eclipse primario y del secundario en todas las
curvas de luz, usando los P_orb, t0 y duraciones de `priors_completosv1.csv`. El resultado
se guarda en **`priors_v2.csv`**, con las mismas columnas, para que
`4_pipeline_completo.ipynb` lo use en lugar de la versión anterior. `priors_completosv1.csv`
no se modifica, y sus valores quedan en las columnas `*_v1`.

Contenido:

1. Por qué el estimador anterior subestima la profundidad (sección 1).
2. El estimador nuevo y su justificación (sección 2).
3. Corrección de periodos mal duplicados (sección 3).
4. Medición, figuras de control y catálogo `priors_v2.csv`.

## 1) Por qué fallaba el estimador anterior

Sea $F(\phi)$ el flujo normalizado en función de la fase orbital $\phi\in[-\tfrac12,\tfrac12)$,
con el eclipse primario centrado en $\phi=0$:

$$F(\phi) = 1-\delta(\phi),\qquad D \equiv \max_\phi \delta(\phi) = 1 - F_{\min}.$$

$D$ es la **profundidad**: la caída de flujo en el centro del eclipse. Es la cantidad que
entra en $D=(R_2/R_1)^2$.

El estimador anterior (`priors_chidos` v3) era

$$\hat D_{v1} = \operatorname{med}\{F_i : 0.15<\phi_i<0.35\} - \operatorname{med}\{F_i : |\phi_i|<h\},\qquad h=0.03,$$

es decir, una ventana de semiancho **fijo** en fase. Sea $W = \tau/P$ el ancho total del
eclipse en fase ($\tau$ = duración, $P$ = periodo). Con los puntos distribuidos
uniformemente en fase, hay dos casos:

**(a) Eclipse más angosto que la ventana, $W<2h$.** La fracción de puntos de la ventana que
sí están en eclipse es $W/2h$. La mediana es el valor que deja la mitad de los puntos a cada
lado, así que si $W/2h<\tfrac12$ (es decir, $W<h=0.03$) **más de la mitad de la ventana está
fuera de eclipse y la mediana es ≈ 1**:

$$W<h \;\Rightarrow\; \hat D_{v1}\approx 0 .$$

Esto pasa en los sistemas de periodo largo. En TIC 276662451, $W=0.018$ y se midió
$\hat D_{v1}=0.006$ cuando el eclipse baja ~0.29.

**(b) Eclipse en V.** Aproximando el eclipse por un triángulo,
$\delta(\phi)=D\,(1-2|\phi|/W)$ para $|\phi|<W/2$. Si la ventana cae dentro del eclipse
($h\le W/2$), $|\phi|$ es uniforme en $[0,h]$ y $\delta$ es lineal en $|\phi|$, así que
$\delta$ es uniforme entre $D(1-2h/W)$ y $D$. Su mediana es

$$\operatorname{med}\,\delta = D\left(1-\frac{h}{W}\right).$$

Por ejemplo, con $W=0.057$ (TIC 11300044) la mediana es $0.47\,D$. El eclipse real no es un
triángulo perfecto (tiene el fondo algo redondeado), por eso se observó $0.69\,D$, pero el
sesgo va siempre en la misma dirección: **$\hat D_{v1}$ subestima la profundidad cuando $h$
no es mucho menor que $W$.**

La mediana es el problema porque es un estimador de *tendencia central* de la ventana. Lo
que se quiere es el *extremo* del perfil. La ventana tiene que escalar con $W$ y ser mucho
más angosta que el eclipse.

## 2) Estimador nuevo

### 2.1 Detrending sin los eclipses
wotan (biweight, ventana $L=0.55$ d) estima la tendencia $T(t)$ como una localización robusta
dentro de una ventana deslizante. Si el eclipse ocupa una fracción $\tau/L$ de esa ventana
(0.12/0.55 ≈ 22 % en eclipses típicos), el biweight se "hunde" parcialmente hacia el eclipse,
y al dividir $f/T$ parte de la profundidad se pierde. Para evitarlo, la tendencia se calcula
**solo con los puntos fuera de eclipse**:

$$\mathcal{O}_T=\{i:\ |\phi_i|>W_m,\ \ |\phi_i-\phi_2|>W_m\},\quad W_m=\min(W,0.2),$$

y $T$ se interpola linealmente sobre los eclipses. El flujo normalizado es $F_i=f_i/T(t_i)$.
$\phi_2$ es la fase del eclipse secundario. En una primera pasada se supone $\phi_2=0.5$; si se
detecta el secundario en otra fase (§2.5), la tendencia se recalcula enmascarándolo donde
realmente está. Si no se hace, el biweight se hunde en el secundario y al dividir aparece un
"bulto" falso junto a él.

### 2.2 Línea base y ruido
Línea base: $\hat B=\operatorname{med}\{F_i: |\phi_i|>e,\ |\phi_i-\phi_2|>e\}$, con
$e=\max\!\left(0.75W,\ \min(1.5W,\,0.2)\right)$. Normalmente $e=1.5W$ (tres semianchos de
margen). En eclipses anchos $e$ se acota a 0.2 para que quede región fuera de eclipse, pero
nunca baja de $0.75W$ (1.5 semianchos).

Ruido por punto, con un estimador robusto que no depende de dónde estén los eclipses
(diferencias sucesivas):

$$\hat\sigma = \frac{1.4826\ \mathrm{MAD}(F_{i+1}-F_i)}{\sqrt2}.$$

El factor 1.4826 convierte el MAD en desviación estándar para ruido gaussiano. El $\sqrt2$
aparece porque la diferencia de dos puntos independientes tiene varianza $2\sigma^2$.

### 2.3 Fondo del eclipse con una ventana proporcional al ancho
Para una fracción $\alpha$ del ancho del eclipse se define

$$m(c;\alpha)=\operatorname{med}\{F_i:\ |\phi_i-c|<\alpha W/2\}$$

y el fondo del eclipse es $F_{\min}=\min_c m(c;\alpha)$. La búsqueda tiene dos etapas:

1. **Ubicación**: con una ventana ancha, $\alpha_0=0.3$, se barre $c$ en el rango de búsqueda
   (§2.5) y se toma $c_0=\arg\min_c m(c;\alpha_0)$. Con ventana ancha todas las posiciones
   tienen puntos suficientes, así que un eclipse mal muestreado no se salta.
2. **Refinamiento**: alrededor de $c_0$, en $[c_0-\alpha_0W/2,\ c_0+\alpha_0W/2]$, se repite la
   búsqueda con la $\alpha$ más pequeña posible (ver abajo) y se obtiene $c^\ast$ y $F_{\min}$.

Para el primario el rango de búsqueda es $[-W/2,\,W/2]$, así que un $t_0$ ligeramente
corrido no importa. El centro encontrado da un $t_0$ refinado: $t_0^{v2}=t_0+c^\ast P$.

**Sesgo.** Con el mismo modelo en V de la sección 1(b), pero con ventana de semiancho
$\alpha W/2$,

$$\operatorname{med}\,\delta = D\left(1-\frac{\alpha}{2}\right)
\;\Rightarrow\; \frac{D-\hat D}{D}\le\frac{\alpha}{2}.$$

Ese es el peor caso (V pura). En un eclipse de fondo plano (eclipse total) el sesgo es
**cero**, porque $\delta=D$ en toda la ventana. La columna `sesgo_max_primario` guarda la cota
$\alpha/2$ de cada medición.

**Elección de α.** Una $\alpha$ pequeña reduce el sesgo pero deja menos puntos. En el
refinamiento se usa la **menor** $\alpha$ de $\{0.05,0.1,0.15,0.2,0.3\}$ con la que la ventana
tenga al menos $N_{\min}=5$ puntos. Si ninguna lo logra (eclipse muy corto con cadencia de
30 min), se usa directamente la ventana de ubicación ($\alpha_0=0.3$, al menos 3 puntos), con
cota de sesgo $\alpha_0/2=15\,\%$. En S98 (cadencia de 200 s) casi siempre alcanza $\alpha=0.05$ (sesgo
≤ 2.5 %). En S5/S6 (30 min) suele necesitar 0.1–0.3 (sesgo ≤ 5–15 % en el peor caso).

**Tiempo de exposición.** Con cadencia de 30 min cada punto es un promedio de 30 min de
flujo. Si $t_{\exp}$ no es pequeño frente a la duración $\tau$, el mínimo observado ya viene
"suavizado" y $D$ queda algo subestimado aunque el estimador sea perfecto. La columna
`texp_sobre_duracion` $=t_{\exp}/\tau$ dice cuándo esto importa (≳ 0.2). En starry se puede
corregir con `starry.System(..., texp=...)`.

### 2.4 Profundidad e incertidumbre

$$\hat D=\hat B-F_{\min},\qquad
\sigma_{\hat D}=\sqrt{\sigma_B^2+\sigma_{\min}^2},\qquad
\sigma_B=\frac{1.2533\,\hat\sigma}{\sqrt{n_B}},\quad \sigma_{\min}=\frac{1.2533\,\hat\sigma}{\sqrt{n_{\min}}}.$$

$1.2533=\sqrt{\pi/2}$ es la razón entre el error estándar de la mediana y el de la media
para ruido gaussiano. $\sigma_{\hat D}$ **no** incluye el sesgo sistemático de 2.3, que se
reporta aparte.

Tomar el mínimo sobre varios centros $c$ también sesga $\hat D$ un poco hacia arriba
(se elige la ventana más baja entre varias ruidosas). Como las ventanas se traslapan casi
por completo, el número de intentos independientes es ~$1/\alpha$ y el efecto es del orden
de $1$–$2\,\sigma_{\min}$, despreciable frente a $D$ en todos los eclipses significativos.

### 2.5 Eclipse secundario y excentricidad
En una órbita excéntrica el secundario **no** cae en fase 0.5. A primer orden en $e$
(Sterne 1939),

$$\Delta\phi\equiv\phi_2-\tfrac12\approx\frac{2}{\pi}\,e\cos\omega
\quad\Longrightarrow\quad e\cos\omega\approx\frac{\pi}{2}\,\Delta\phi .$$

Por ejemplo, en TIC 276662451 el secundario está en $\phi_2\approx0.565$, lo que da
$e\cos\omega\approx0.10$: ahí la suposición de órbita circular no vale.

Por eso el secundario se mide de dos formas:

- **búsqueda global**: el estimador de §2.3 con el rango de ubicación en toda la órbita fuera
  del primario, $c\in[e_g,\ 1-e_g]$ con $e_g=\min(1.5W,\,0.2)$. Si el refinamiento termina a
  menos de $e_g$ del primario, se descarta, porque lo que se encontró es el ala del propio
  primario;
- **búsqueda local**: rango $[0.5-W/2,\ 0.5+W/2]$, como si la órbita fuera circular.

**Criterio de detección.** En la búsqueda global el mínimo se toma sobre muchas posiciones, y
la curva tiene ruido correlacionado (manchas, sistemáticos) que $\hat\sigma$ de §2.2 no ve,
porque mide solo el ruido punto a punto. Por eso la significancia se mide contra la
**distribución empírica** de las medianas de ventana en la etapa de ubicación:

$$\sigma_{\rm emp}=1.4826\ \mathrm{MAD}\{m(c;\alpha_0)\},\qquad c\in[W,\,1-W].$$

El MAD es robusto: el propio eclipse ocupa pocas posiciones y no infla $\sigma_{\rm emp}$. Si
solo hubiera ruido, el mínimo de $N$ medianas independientes caería en promedio a
$\approx\sqrt{2\ln N}\,\sigma_{\rm emp}$ de la línea base. Con $N\sim1/(\alpha_0W)\lesssim 200$
eso es $\lesssim3.3\,\sigma_{\rm emp}$. El secundario global se acepta solo si

$$\hat B-m(c_0;\alpha_0)>5\,\sigma_{\rm emp}\quad\text{y}\quad \hat D_2>5\,\sigma_{\hat D_2}.$$

Si se acepta, se reportan su profundidad (refinada), su fase $\phi_2$ y $e\cos\omega$. Si no,
se reporta la medición local en 0.5 y $e\cos\omega$ queda indefinido.

El valor de $e\cos\omega$ es una **cota inferior** de la excentricidad, $e\ge|e\cos\omega|$.
$e\sin\omega$ se podría estimar del cociente de duraciones de los dos eclipses, pero no se
intenta aquí.

### 2.6 Casos que no son un eclipse "limpio"
- `eclipse_ancho`: si $W>0.25$, BLS no encontró un eclipse separado (contacto, elipsoidal o
  falso positivo). La línea base se toma como el percentil 90 de $F$ y la medición se marca
  para revisión.
- `no_significativo`: si $\hat D<3\,\sigma_{\hat D}$.

### 2.8 Duración robusta
La duración de BLS a veces se queda en un valor absurdo en un sector: en TIC 50897998 S32 da
0.056 d, contra 0.21–0.25 d en los otros sectores. Con ella $W$ es 4 veces menor, la máscara no
cubre los eclipses y la línea base incluye sus alas. Como la duración es una propiedad del
sistema, si la de un sector se aleja más de 40 % de la mediana del TIC se usa la mediana
(`Duration_corregida`; el valor de BLS queda en `Duration_v1`). No se aplica a sectores donde BLS
falló (duración original $>0.25$ en fase, §2.6): ahí el periodo tampoco es confiable, y ese sector
debe quedar fuera de la prueba par-impar.

### 2.9 El eclipse primario es el más profundo
BLS pone $t_0$ en el eclipse que más favorece su estadístico, que a veces es el **menos** profundo.
Por convención (y porque starry pone la secundaria delante de la primaria en $t_0$), el primario
es el eclipse más profundo. Si $\hat D(\phi_2)-\hat D(0)>3\sigma$, se mueve $t_0\to t_0+\phi_2P$ y se vuelve
a medir, lo que intercambia primario y secundario (`primario_intercambiado`).

### 2.10 $e\cos\omega$ y su incertidumbre
$e\cos\omega$ se calcula con el desfase entre los **dos centros medidos**:

$$e\cos\omega\simeq\frac{\pi}{2}\left[(c_2-c_1)\bmod 1-\tfrac12\right],$$

con $c_1$ y $c_2$ los centros del primario y del secundario de §2.3. Antes se usaba $c_2$ respecto de
$t_0$, sin restar $c_1$: si $t_0$ está corrido respecto del mínimo real del primario (hasta ~0.007 en
fase en algunos sectores), ese corrimiento pasaba entero a $e\cos\omega$, en un factor $\pi/2$.

**Error por bootstrap.** Se remuestrean los puntos de la curva con reemplazo ($N=100$), se vuelven
a medir los dos centros (búsqueda en $c\pm W/4$) y se toma la desviación estándar de $c_2-c_1$. A eso
se suma en cuadratura el error de cuantización de la malla de centros: paso $\Delta c=\alpha W/4$,
error $\sqrt2\,\Delta c/\sqrt{12}$ para la diferencia de dos centros. El resultado es
$\sigma_{e\cos\omega}=\tfrac{\pi}{2}\,\sigma_{\Delta\phi}$.

El bootstrap solo captura el ruido de los puntos. Los sistemáticos (manchas que desplazan los
mínimos, residuos del detrending) se evalúan al combinar sectores: si los sectores de un sistema
no concuerdan dentro de sus errores, el error combinado se infla (ver `explore_priors_v2`).

### 2.7 Relación con el radio
Se mantiene $D=(R_2/R_1)^2$, es decir $R_2=R_1\sqrt{D}$. Esta relación supone que la
secundaria aporta poca luz. En general, la profundidad del primario es
$D\approx\ell_1\,(R_2/R_1)^2$ con $\ell_1$ la fracción de luz de la primaria, así que $R_2$
queda subestimado por un factor $\sqrt{\ell_1}$ cuando la secundaria es brillante.

## 3) Periodo orbital: ¿P o 2P? Prueba par-impar por sistema

BLS encuentra un periodo $P$, pero no puede distinguir entre dos situaciones que dan la misma curva
plegada:

- **(a)** el periodo real es $P$: en cada ciclo hay un eclipse primario (y quizá un secundario en otra fase);
- **(b)** el periodo real es $2P$ y los dos eclipses son **parecidos**. Al plegar con $P$, el primario y el
  secundario caen uno encima del otro en fase 0, y en fase 0.5 "no hay secundario".

Pasa también al revés: si el periodo real es $P$ y se pliega con $2P$, en fase 0 y 0.5 caen primarios
alternados, que tienen la **misma** profundidad.

**Cuándo hace falta la prueba.** Si con $P_{\rm base}$ ya se detecta un secundario (§2.5) en algún
sector, hay un segundo eclipse *distinto* en otra fase y eso demuestra el caso (a). Solo cuando **ningún**
sector muestra secundario con $P_{\rm base}$ queda abierto el caso (b), y entonces se hace la prueba.

**Prueba par-impar.** Se pliega con $2P$ y se comparan los dos eclipses:

$$z=\frac{\hat D(0)-\hat D(\phi_2)}{\sqrt{\sigma_{\hat D(0)}^2+\sigma_{\hat D(\phi_2)}^2+(f_s\hat D(0))^2+(f_s\hat D(\phi_2))^2}} ,\qquad f_s=0.03 .$$

$f_s$ es un **piso sistemático** del 3 % de la profundidad. $\sigma_{\hat D}$ solo incluye el ruido
blanco, pero entre eclipses individuales la profundidad varía algunos % por manchas cruzadas,
contaminación y detrending. Sin ese piso, en eclipses profundos cualquier variación real da $|z|$ de
cientos. Además se exige que el segundo eclipse con $2P$ esté en $|\phi_2-0.5|<W/2$, que es donde cae
el otro eclipse en el caso (b) con órbita circular.

En (a), los dos "eclipses" son el mismo primario en ciclos alternos, así que $z$ es solo ruido,
$z\sim\mathcal N(0,1)$. En (b) son el primario y el secundario, que tienen brillos superficiales
distintos y por eso $|z|\gg1$.

**Se decide por sistema, no por sector.** El periodo orbital es el mismo en todos los sectores, así
que la evidencia se suma:

$$\chi^2=\sum_{\rm sectores} z_s^2,\qquad p=P\!\left(\chi^2_{n}\ge\chi^2\right),$$

con $n$ el número de sectores válidos (sin `eclipse_ancho`). Si $p<0.0027$ (el equivalente a
$3\sigma$), el periodo del sistema es $2P_{\rm base}$; si no, es $P_{\rm base}$. $P_{\rm base}$ es el menor
periodo encontrado entre los sectores (los demás son múltiplos por errores de BLS). Después, todos los
sectores cuyo periodo no coincide con el del sistema se vuelven a medir con él.

Así, un sector con poca señal no puede imponer su periodo a los demás. Esto le pasaba a la regla
anterior: en TIC 249074181, el sector S98 tiene dos eclipses casi idénticos ($z=0.6$), pero S5 y S32
muestran que son distintos ($z=4.9$ y $2.7$), y el periodo correcto es el largo.

**Límite.** Si primaria y secundaria son gemelas exactas, $z\approx0$ incluso en el caso (b), y la
fotometría sola no puede distinguirlas. En ese caso se queda $P_{\rm base}$, y un $P_{\rm rot}=2P_{\rm orb}$ en
todos los sectores es una pista de que el periodo real es $2P$.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wotan import flatten

RUTA_ORION = Path("/home/luciam/Documents/starry/Resultados_Orion")
RUTA_CURVAS = RUTA_ORION / "Light_Curve_Files"
PATH_PRIORS_V1 = RUTA_ORION / "priors_completosv1.csv"
PATH_PRIORS_V2 = RUTA_ORION / "priors_v2.csv"
DIR_FIG = RUTA_ORION / "resultados_pipeline" / "profundidades_v2"
DIR_FIG.mkdir(parents=True, exist_ok=True)

WINDOW_WOTAN = 0.55                       # d, igual que en priors_chidos v3
ALFAS = (0.05, 0.10, 0.15, 0.20, 0.30)    # fracciones del ancho del eclipse (sección 2.3)
N_MIN = 5                                 # puntos mínimos en la ventana del fondo
W_MAX = 0.25                              # arriba de esto no es un eclipse separado (2.6)
TOL_P = 0.01                              # tolerancia para P' ≈ P/2 (sección 3)
MOSTRAR_FIGURAS = True

In [ ]:
def cargar_curva(tic, sector):
    d = pd.read_csv(RUTA_CURVAS / f"tic {tic}_S_{sector}_LC.txt", sep=r"\s+")
    d = d.dropna(subset=["#Time", "Flux_CBV_SS"]).sort_values("#Time").drop_duplicates("#Time")
    return d["#Time"].to_numpy(float), d["Flux_CBV_SS"].to_numpy(float)


def fase(t, t0, P):
    # fase en [-0.5, 0.5), eclipse primario en 0
    return ((t - t0) / P + 0.5) % 1 - 0.5


def dist_fase(ph, c):
    # distancia circular en fase
    return np.abs(((ph - c) + 0.5) % 1 - 0.5)


def normalizar_sin_eclipses(t, f, t0, P, W, phi2=0.5):
    # sección 2.1: tendencia biweight solo con puntos fuera de eclipse
    ph = fase(t, t0, P)
    Wm = min(W, 0.2)
    fuera = (dist_fase(ph, 0.0) > Wm) & (dist_fase(ph, phi2) > Wm)
    try:
        if fuera.sum() < 100:
            raise ValueError
        _, tr = flatten(t[fuera], f[fuera], method="biweight", window_length=WINDOW_WOTAN,
                        break_tolerance=0.5, return_trend=True)
        ok = np.isfinite(tr)
        T = np.interp(t, t[fuera][ok], tr[ok])
    except ValueError:
        T = np.full_like(f, np.nanmedian(f))   # la máscara tapa casi todo: normalización global
    return f / T, T


def ruido_por_punto(F):
    # sección 2.2
    return 1.4826 * np.nanmedian(np.abs(np.diff(F) - np.nanmedian(np.diff(F)))) / np.sqrt(2)


ALFA_0 = 0.3   # ventana de la etapa de ubicación (§2.3)


def medianas_ventana(ph, F, centros, media_ventana, n_min):
    m = np.full(len(centros), np.nan)
    n = np.zeros(len(centros), dtype=int)
    for j, c in enumerate(centros):
        sel = dist_fase(ph, c) < media_ventana
        n[j] = sel.sum()
        if n[j] >= n_min:
            m[j] = np.nanmedian(F[sel])
    return m, n


def fondo_eclipse(ph, F, centro, semirango, W):
    # §2.3, etapa 1: ubicación con ventana ancha
    mv0 = ALFA_0 * W / 2
    centros = np.arange(centro - semirango, centro + semirango + 1e-12, max(mv0 / 4, 1e-4))
    m0, n0 = medianas_ventana(ph, F, centros, mv0, 3)
    if np.all(np.isnan(m0)):
        return None
    j0 = int(np.nanargmin(m0))
    c0 = centros[j0]
    sigma_emp = 1.4826 * np.nanmedian(np.abs(m0 - np.nanmedian(m0)))
    # §2.3, etapa 2: refinamiento con la menor alfa posible
    for alfa in ALFAS:
        mv = alfa * W / 2
        cs = np.arange(c0 - mv0, c0 + mv0 + 1e-12, max(mv / 2, 1e-4))
        m, n = medianas_ventana(ph, F, cs, mv, N_MIN)
        if not np.all(np.isnan(m)):
            j = int(np.nanargmin(m))
            return dict(F_min=m[j], n=int(n[j]), centro=((cs[j] + 0.5) % 1) - 0.5, alfa=alfa,
                        m0_min=m0[j0], sigma_emp=sigma_emp)
    # respaldo: eclipse muy poco muestreado -> se usa la ventana de ubicación (>= 3 puntos)
    return dict(F_min=m0[j0], n=int(n0[j0]), centro=((c0 + 0.5) % 1) - 0.5, alfa=ALFA_0,
                m0_min=m0[j0], sigma_emp=sigma_emp)


def medir(t, f, P, t0, dur):
    W = dur / P
    ph = fase(t, t0, P)
    ancho = W > W_MAX
    phi2 = 0.5
    for paso in (1, 2):
        F, T = normalizar_sin_eclipses(t, f, t0, P, W, phi2)
        sigma = ruido_por_punto(F)
        if ancho:
            B = float(np.nanpercentile(F, 90))
            nB = int((F >= B).sum())
        else:
            e = max(0.75 * W, min(1.5 * W, 0.2))   # sección 2.2
            mB = (dist_fase(ph, 0.0) > e) & (dist_fase(ph, phi2) > e)
            B, nB = float(np.nanmedian(F[mB])), int(mB.sum())
        sB = 1.2533 * sigma / np.sqrt(max(nB, 1))

        def resultado(r):
            if r is None:
                return None
            return dict(D=B - r["F_min"], err=np.hypot(sB, 1.2533 * sigma / np.sqrt(r["n"])),
                        alfa=r["alfa"], n=r["n"], centro=r["centro"],
                        signif_emp=(B - r["m0_min"]) / r["sigma_emp"] if r["sigma_emp"] > 0 else np.inf)

        Wb = min(W, 0.5)
        prim = resultado(fondo_eclipse(ph, F, 0.0, Wb / 2, Wb))
        sec_local = resultado(fondo_eclipse(ph, F, 0.5, Wb / 2, Wb))
        e_glob = min(1.5 * Wb, 0.2)   # zona excluida alrededor del primario (§2.5)
        sec_global = None if ancho else resultado(fondo_eclipse(ph, F, 0.5, max(0.5 - e_glob, Wb / 2), Wb))
        detectado = (sec_global is not None and sec_global["D"] > 5 * sec_global["err"]
                     and sec_global["signif_emp"] > 5
                     and dist_fase(np.array([sec_global["centro"]]), 0.0)[0] > e_glob)   # §2.5
        sec = sec_global if detectado else sec_local
        nuevo_phi2 = (sec_global["centro"] % 1) if detectado else 0.5
        if paso == 2 or abs(nuevo_phi2 - phi2) < 1e-9:
            break
        phi2 = nuevo_phi2    # segunda pasada: tendencia enmascarando el secundario donde está

    out = dict(W=W, sigma_punto=sigma, B=B, eclipse_ancho=ancho, secundario_detectado=detectado,
               signif_emp_secundario=sec_global["signif_emp"] if sec_global else np.nan,
               fase_secundario=(sec["centro"] % 1) if sec else np.nan,
               # §2.10: desfase del secundario respecto del primario MEDIDO (no respecto de t0)
               ecosw=(np.pi / 2) * (((sec["centro"] - prim["centro"]) % 1) - 0.5)
               if (detectado and prim is not None) else np.nan)
    for nombre, r in (("primario", prim), ("secundario", sec)):
        if r is None:
            out.update({f"D_{nombre}": np.nan, f"D_{nombre}_err": np.nan, f"alfa_{nombre}": np.nan,
                        f"n_{nombre}": 0, f"centro_{nombre}": np.nan})
        else:
            out.update({f"D_{nombre}": r["D"], f"D_{nombre}_err": r["err"], f"alfa_{nombre}": r["alfa"],
                        f"n_{nombre}": r["n"], f"centro_{nombre}": r["centro"]})
    return out, F, ph

## 4) Medición en todas las curvas

In [ ]:
v1 = pd.read_csv(PATH_PRIORS_V1)
# duración robusta (§2.8): si la de BLS se aleja >40 % de la mediana del sistema, se usa la mediana
dur_med_tic = v1.groupby("TIC")["Duration"].median()
filas, curvas = [], {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for _, p in v1.iterrows():
        tic, s = int(p["TIC"]), int(p["Sector"])
        t, f = cargar_curva(tic, s)
        texp = np.median(np.diff(t))
        dur = p["Duration"]
        # no se corrige un sector donde BLS falló (eclipse "ancho" con la duración original, §2.6)
        dur_corr = abs(dur / dur_med_tic[tic] - 1) > 0.4 and dur / p["P_orb"] <= W_MAX
        if dur_corr:
            print(f"TIC {tic} S{s}: duración BLS {dur:.4f} d -> mediana del sistema {dur_med_tic[tic]:.4f} d")
            dur = dur_med_tic[tic]
        m, F, ph = medir(t, f, p["P_orb"], p["t0"], dur)
        curvas[(tic, s)] = (t, f)
        filas.append(dict(TIC=tic, Sector=s, P=p["P_orb"], t0=p["t0"], Duration=dur, Duration_corregida=dur_corr,
                          texp_sobre_duracion=texp / dur, **m))
med = pd.DataFrame(filas)
print(f"{len(med)} curvas medidas")

In [ ]:
# -------- sección 3: prueba par-impar por sistema --------
from scipy.stats import chi2 as dist_chi2

P_SIGNIF = 0.0027
F_SIS = 0.03    # piso sistemático de la profundidad por eclipse (manchas, detrending)
med["P_corregido"] = False
filas_pi = []
for tic, g in med.groupby("TIC"):
    g_ok = g.loc[~g["eclipse_ancho"]]
    if g_ok.empty:
        continue
    P_base = g_ok["P"].min()
    # ¿algún sector muestra secundario con P_base? -> caso (a) demostrado, no hace falta la prueba
    con_sec = False
    for i, r in g_ok.iterrows():
        t, f = curvas[(r["TIC"], r["Sector"])]
        mb, _, _ = medir(t, f, P_base, r["t0"], r["Duration"])
        con_sec = con_sec or bool(mb["secundario_detectado"])
    z = []
    if not con_sec:
        for i, r in g_ok.iterrows():
            t, f = curvas[(r["TIC"], r["Sector"])]
            m, _, _ = medir(t, f, 2 * P_base, r["t0"], r["Duration"])
            D0, D2 = m["D_primario"], m["D_secundario"]
            s = np.sqrt(m["D_primario_err"] ** 2 + m["D_secundario_err"] ** 2 + (F_SIS * D0) ** 2 + (F_SIS * D2) ** 2)
            cerca_media = abs(m["fase_secundario"] - 0.5) < m["W"] / 2 if np.isfinite(m["fase_secundario"]) else False
            if np.isfinite(D0) and np.isfinite(D2) and s > 0 and cerca_media:
                z.append((D0 - D2) / s)
    chi2_v = float(np.sum(np.square(z))) if z else np.nan
    p = float(dist_chi2.sf(chi2_v, len(z))) if z else np.nan
    P_sis = 2 * P_base if (z and p < P_SIGNIF) else P_base
    filas_pi.append(dict(TIC=tic, P_base=P_base, secundario_con_P_base=con_sec, z_sectores=", ".join(f"{v:+.1f}" for v in z), chi2=chi2_v,
                         n=len(z), p=p, P_sistema=P_sis))
    for i, r in g.iterrows():
        if r["eclipse_ancho"] or abs(r["P"] / P_sis - 1) < TOL_P:
            continue
        t, f = curvas[(r["TIC"], r["Sector"])]
        m, _, _ = medir(t, f, P_sis, r["t0"], r["Duration"])
        print(f"TIC {tic} S{r['Sector']}: P {r['P']:.6f} -> {P_sis:.6f} d  (par-impar del sistema: "
              f"chi2 = {chi2_v:.1f}, n = {len(z)}, p = {p:.1e})")
        for k, v in m.items():
            med.at[i, k] = v
        med.at[i, "P"] = P_sis
        med.at[i, "P_corregido"] = True

par_impar = pd.DataFrame(filas_pi)
par_impar.loc[~par_impar["secundario_con_P_base"]].round(4)

## 4b) Periodo de rotación con la efeméride corregida

El $P_{\rm rot}$ del catálogo v1 se calculó con Lomb-Scargle fuera de eclipse, pero con la
máscara construida con el $P_{\rm orb}$ de v1. Donde ese periodo estaba mal, la máscara dejaba
pasar eclipses y el periodograma se contaminaba. Aquí se recalcula con la efeméride v2
($P_{\rm orb}$, $t_0^{v2}$ y la fase real del secundario $\phi_2$):

1. **Máscara**: se quitan los puntos con $|\phi|<W_r$ o $|\phi-\phi_2|<W_r$, con
   $W_r=\min(W,\,0.2)$. El semiancho de la máscara es igual al ancho total del eclipse, así
   que se quita el doble de lo que dura cada eclipse.
2. **Flujo sin aplanar**: se usa `Flux_CBV_SS / mediana`. La modulación por manchas *es* la
   tendencia que wotan le quita a la curva, así que no se puede usar el flujo aplanado.
3. **Lomb-Scargle** en $P\in[0.2,\,15]$ d, con una malla uniforme en frecuencia y sobremuestreo
   ×10 respecto a la resolución $1/T$ ($T$ = duración del sector).

**Incertidumbre.** Se usa la semianchura a media altura del pico, $\delta f$, y se propaga a
periodo con $P=1/f$:

$$\sigma_P\approx P^2\,\delta f .$$

Es una cota conservadora: el ancho del pico lo fija sobre todo la duración del sector, porque
$\delta f\sim 1/T$.

**Significancia.** Probabilidad de falsa alarma de Baluev (2008) para la potencia máxima. En TESS
casi siempre da ≈ 0, porque los sistemáticos también son "señal". Por eso importa más que el
periodo se repita entre sectores. Para eso se guardan los 3 picos más altos de cada sector
(`P_rot_picos`, `P_rot_picos_pot`), y `explore_priors_v2.ipynb` elige el **periodo de
consenso** entre sectores.

**Armónicos del periodo orbital.** Si $P_{\rm rot}/P_{\rm orb}$ cae a menos de 3 % de 1/2, 1 o 2,
se marca `rot_armonico`. En binarias cercanas, la variación elipsoidal (con periodo
$P_{\rm orb}/2$) y la reflexión (con periodo $P_{\rm orb}$) también producen modulación fuera de
eclipse. Un $P_{\rm rot}=P_{\rm orb}$ puede ser sincronización real **o** reflexión: conviene
revisar la forma de la modulación (las manchas cambian entre sectores; la reflexión no).

In [ ]:
from astropy.timeseries import LombScargle


def periodo_rotacion(t, f, P, t0, W, phi2):
    ph = fase(t, t0, P)
    Wr = min(W, 0.2)
    fuera = (dist_fase(ph, 0.0) > Wr) & (dist_fase(ph, phi2) > Wr)
    if fuera.sum() < 50:
        return dict(P_rot=np.nan, P_rot_err=np.nan, P_rot_FAP=np.nan, P_rot_potencia=np.nan,
                    P_rot_picos="", P_rot_picos_pot="")
    tt, ff = t[fuera], f[fuera] / np.nanmedian(f[fuera])
    T = tt.max() - tt.min()
    fmin, fmax = 1 / 15.0, 1 / 0.2
    frec = np.linspace(fmin, fmax, int(10 * T * (fmax - fmin)))
    ls = LombScargle(tt, ff)
    pot = ls.power(frec)
    j = int(np.argmax(pot))
    medio = pot[j] / 2
    izq = j
    while izq > 0 and pot[izq] > medio:
        izq -= 1
    der = j
    while der < len(pot) - 1 and pot[der] > medio:
        der += 1
    hwhm = (frec[der] - frec[izq]) / 2
    Prot = 1 / frec[j]
    # los 3 picos locales más altos (separados por más de 2 HWHM), para el consenso entre sectores
    loc = np.where((pot[1:-1] > pot[:-2]) & (pot[1:-1] >= pot[2:]))[0] + 1
    picos = []
    for k in loc[np.argsort(pot[loc])[::-1]]:
        if all(abs(frec[k] - frec[m]) > 2 * hwhm for m in picos):
            picos.append(k)
        if len(picos) == 3:
            break
    return dict(P_rot=Prot, P_rot_err=Prot ** 2 * hwhm,
                P_rot_picos=";".join(f"{1 / frec[k]:.5f}" for k in picos),
                P_rot_picos_pot=";".join(f"{pot[k]:.4f}" for k in picos),
                P_rot_FAP=float(ls.false_alarm_probability(pot[j], method="baluev",
                                                            minimum_frequency=fmin, maximum_frequency=fmax)),
                P_rot_potencia=float(pot[j]))


filas_rot = []
for i, r in med.iterrows():
    t, f = curvas[(r["TIC"], r["Sector"])]
    t0c = r["t0"] + (r["centro_primario"] if np.isfinite(r["centro_primario"]) else 0.0) * r["P"]
    phi2 = r["fase_secundario"] if r["secundario_detectado"] else 0.5
    filas_rot.append(periodo_rotacion(t, f, r["P"], t0c, r["W"], phi2))
rot = pd.DataFrame(filas_rot, index=med.index)
for c in rot.columns:
    med[c] = rot[c]

cociente = med["P_rot"] / med["P"]
med["rot_armonico"] = [next((f"{k:g}xP_orb" for k in (0.5, 1, 2) if abs(q / k - 1) < 0.03), "")
                       for q in cociente]
med[["TIC", "Sector", "P", "P_rot", "P_rot_err", "P_rot_FAP", "rot_armonico"]].round(4)

In [ ]:
# -------- §2.9: el eclipse primario es el más profundo --------
med["primario_intercambiado"] = False
for i, r in med.iterrows():
    d1, d2 = r["D_primario"], r["D_secundario"]
    s = np.hypot(r["D_primario_err"], r["D_secundario_err"])
    if np.isfinite(d1) and np.isfinite(d2) and d2 - d1 > 3 * s and not r["eclipse_ancho"]:
        phi2 = r["fase_secundario"] if np.isfinite(r["fase_secundario"]) else 0.5
        t, f = curvas[(r["TIC"], r["Sector"])]
        t0_nuevo = r["t0"] + phi2 * r["P"]
        m, _, _ = medir(t, f, r["P"], t0_nuevo, r["Duration"])
        print(f"TIC {r['TIC']} S{r['Sector']}: el eclipse en fase {phi2:.3f} es más profundo "
              f"({d2:.4f} > {d1:.4f}); t0 se mueve a ese eclipse (nuevo D_primario = {m['D_primario']:.4f})")
        for k, v in m.items():
            med.at[i, k] = v
        med.at[i, "t0"] = t0_nuevo
        med.at[i, "primario_intercambiado"] = True

# -------- §2.10: incertidumbre de e cos w por bootstrap --------
N_BOOT = 100
rng = np.random.default_rng(42)
med["ecosw_err"] = np.nan
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for i, r in med.iterrows():
        if not r["secundario_detectado"] or r["eclipse_ancho"]:
            continue
        t, f = curvas[(r["TIC"], r["Sector"])]
        _, F, ph = medir(t, f, r["P"], r["t0"], r["Duration"])
        W = min(r["W"], 0.5)
        c1, c2 = r["centro_primario"], r["fase_secundario"]
        difs = []
        for _ in range(N_BOOT):
            k = rng.integers(0, len(F), len(F))
            a = fondo_eclipse(ph[k], F[k], c1, W / 4, W)
            b = fondo_eclipse(ph[k], F[k], c2, W / 4, W)
            if a is not None and b is not None:
                difs.append(((b["centro"] - a["centro"]) % 1) - 0.5)
        paso = max(np.nanmax([r["alfa_primario"], r["alfa_secundario"]]) * W / 4, 1e-4)
        if len(difs) > 10:
            sig_dphi = np.hypot(np.std(difs), np.sqrt(2) * paso / np.sqrt(12))
            med.at[i, "ecosw_err"] = np.pi / 2 * sig_dphi
print(f"e cos w con error en {med['ecosw_err'].notna().sum()} sectores; "
      f"mediana del error = {med['ecosw_err'].median():.4f}")

# -------- calidad de cada medición --------
def calidad(r):
    notas = []
    if r["eclipse_ancho"]:
        notas.append("eclipse_ancho")
    if not (r["D_primario"] > 3 * r["D_primario_err"]):
        notas.append("no_significativo")
    if r["P_corregido"]:
        notas.append("P_corregido")
    if r["texp_sobre_duracion"] > 0.2:
        notas.append("texp_largo")
    if r["Duration_corregida"]:
        notas.append("duracion_mediana")
    if r["primario_intercambiado"]:
        notas.append("primario_intercambiado")
    return ",".join(notas) or "ok"


med["Calidad_depth"] = med.apply(calidad, axis=1)
med["sesgo_max_primario"] = med["alfa_primario"] / 2
med["sesgo_max_secundario"] = med["alfa_secundario"] / 2
med["t0_v2"] = med["t0"] + med["centro_primario"].fillna(0) * med["P"]

comp = v1[["TIC", "Sector", "Depth_primary"]].merge(med[["TIC", "Sector", "D_primario", "Calidad_depth"]])
comp["cociente_v2_v1"] = comp["D_primario"] / comp["Depth_primary"]
print(med["Calidad_depth"].value_counts().to_string())
print("\nD_v2 / D_v1 (solo 'ok'):", comp.loc[comp["Calidad_depth"] == "ok", "cociente_v2_v1"].describe().round(3).to_dict())
comp.round(4)

## 5) Figuras de control

Una figura por TIC, con un panel por sector y un par (primario | secundario). Cada panel
muestra el flujo normalizado sin eclipses (§2.1) plegado en fase, centrado en el eclipse (el
secundario se centra en la fase donde se detectó), y tres líneas:
$1-D_{v1}$ (roja), $1-\hat D_{v2}$ (azul) y la línea base $\hat B$ (gris). La franja naranja es
la ventana de ±0.03 del estimador anterior y la verde, la ventana $\alpha W/2$ del nuevo.

In [ ]:
for tic, g in med.groupby("TIC"):
    g = g.sort_values("Sector")
    fig, axs = plt.subplots(len(g), 2, figsize=(12, 3.2 * len(g)), squeeze=False)
    for fila_ax, (_, r) in zip(axs, g.iterrows()):
        t, f = curvas[(tic, r["Sector"])]
        phi2 = r["fase_secundario"] if r["secundario_detectado"] else 0.5
        F, _ = normalizar_sin_eclipses(t, f, r["t0"], r["P"], r["W"], phi2)
        v = v1.loc[(v1["TIC"] == tic) & (v1["Sector"] == r["Sector"])].iloc[0]
        for ax, nombre, centro, d_v1 in ((fila_ax[0], "primario", 0.0, v["Depth_primary"]),
                                          (fila_ax[1], "secundario", phi2, v["Depth_secondary"])):
            ph = fase(t, r["t0"], r["P"])
            x = ((ph - centro + 0.5) % 1) - 0.5
            ax.plot(x, F, ".", ms=2, color="0.35")
            ax.axhline(r["B"], color="0.6", lw=1)
            if np.isfinite(d_v1):
                ax.axhline(1 - d_v1, color="tab:red", lw=1.3, label=f"v1: {d_v1:.4f}")
            D = r[f"D_{nombre}"]
            if np.isfinite(D):
                ax.axhline(r["B"] - D, color="tab:blue", ls="--", lw=1.3,
                           label=f"v2: {D:.4f} ± {r[f'D_{nombre}_err']:.4f}")
                c = ((r[f"centro_{nombre}"] - centro + 0.5) % 1) - 0.5
                ax.axvspan(c - r[f"alfa_{nombre}"] * r["W"] / 2, c + r[f"alfa_{nombre}"] * r["W"] / 2,
                           color="tab:green", alpha=0.25)
            ax.axvspan(-0.03, 0.03, color="orange", alpha=0.12)
            lim = min(0.5, 2.5 * r["W"] + 0.03)
            ax.set_xlim(-lim, lim)
            extra = f"  fase={phi2:.3f}, e cos w={r['ecosw']:.3f}" if nombre == "secundario" and r["secundario_detectado"] else ""
            ax.set_title(f"S{r['Sector']} {nombre}  P={r['P']:.5f} d  [{r['Calidad_depth']}]{extra}", fontsize=9)
            ax.legend(fontsize=7, loc="lower left")
    fig.suptitle(f"TIC {tic}", fontsize=12)
    fig.tight_layout()
    fig.savefig(DIR_FIG / f"TIC_{tic}_profundidades.png", dpi=110)
    plt.show() if MOSTRAR_FIGURAS else None
    plt.close(fig)

## 6) Catálogo `priors_v2.csv`

Mismas columnas que `priors_completosv1.csv`, con `Depth_primary`, `Depth_secondary`
(y errores), `P_orb`, `t0` y `P_rot` reemplazados por los valores v2 (P_rot con su error, FAP
y la marca `rot_armonico`). Los originales quedan en
`*_v1`, junto con las columnas de diagnóstico (`alfa_*`, `sesgo_max_*`, `n_*`,
`texp_sobre_duracion`, `Calidad_depth`) y las del secundario (`secundario_detectado`,
`fase_secundario`, `ecosw`).

In [ ]:
v2 = v1.copy()
for col in ["Depth_primary", "Depth_primary_err", "Depth_secondary", "Depth_secondary_err", "P_orb", "t0", "P_rot",
            "Duration"]:
    v2[f"{col}_v1"] = v1[col]
m = med.set_index(["TIC", "Sector"])
clave = list(zip(v2["TIC"], v2["Sector"]))
v2["Depth_primary"] = [m.at[k, "D_primario"] for k in clave]
v2["Depth_primary_err"] = [m.at[k, "D_primario_err"] for k in clave]
v2["Depth_secondary"] = [m.at[k, "D_secundario"] for k in clave]
v2["Depth_secondary_err"] = [m.at[k, "D_secundario_err"] for k in clave]
v2["P_orb"] = [m.at[k, "P"] for k in clave]
v2["Duration"] = [m.at[k, "Duration"] for k in clave]
v2["t0"] = [m.at[k, "t0_v2"] for k in clave]
for col in ["alfa_primario", "sesgo_max_primario", "n_primario", "alfa_secundario", "sesgo_max_secundario",
            "n_secundario", "secundario_detectado", "signif_emp_secundario", "fase_secundario", "ecosw",
            "ecosw_err", "centro_primario",
            "texp_sobre_duracion", "sigma_punto", "P_corregido", "Calidad_depth", "Duration_corregida",
            "primario_intercambiado",
            "P_rot", "P_rot_err", "P_rot_FAP", "P_rot_potencia", "rot_armonico", "P_rot_picos", "P_rot_picos_pot"]:
    v2[col] = [m.at[k, col] for k in clave]

v2.to_csv(PATH_PRIORS_V2, index=False)
print(f"Guardado: {PATH_PRIORS_V2}")
v2[["TIC", "Sector", "P_orb", "Depth_primary_v1", "Depth_primary", "Depth_primary_err", "sesgo_max_primario",
    "Depth_secondary_v1", "Depth_secondary", "secundario_detectado", "fase_secundario", "ecosw",
    "Calidad_depth"]].round(4)

## 7) Consistencia de las profundidades entre sectores

La profundidad de un eclipse es una propiedad geométrica del sistema. Salvo que cambie la inclinación
(por ejemplo, por la precesión que induce una tercera estrella), debería salir igual en todos los
sectores. Esta sección compara, para cada TIC, las profundidades primaria y secundaria de sus sectores
válidos (sin `eclipse_ancho`) y marca los sistemas donde difieren más de lo que permiten sus errores.

### 7.1 Error total de cada medición
Cada profundidad tiene dos incertidumbres:

1. **Estadística**, $\sigma_{\rm est}$ (§2.4), que es muy pequeña en sectores de cadencia rápida
   (~$10^{-4}$ en S98).
2. **Sesgo del estimador** (§2.3): la profundidad verdadera está entre $\hat D$ y $\hat D(1+\alpha/2)$,
   según la forma del eclipse (fondo plano o en V). Modelando ese sesgo como una distribución uniforme
   en ese intervalo, la profundidad corregida y su error son

$$\hat D_c=\hat D\left(1+\frac{\alpha}{4}\right),\qquad
\sigma_{\rm ses}=\frac{\alpha\hat D/2}{\sqrt{12}},\qquad
\sigma_{\rm tot}^2=\sigma_{\rm est}^2+\sigma_{\rm ses}^2 .$$

Este término importa sobre todo en S5/S6 (cadencia de 30 min), donde $\alpha$ llega a 0.1–0.3.

### 7.2 Prueba de consistencia
Con pesos $w_s=1/\sigma_{{\rm tot},s}^2$ se calcula el promedio pesado $\bar D=\sum w_s\hat D_{c,s}/\sum w_s$ y

$$\chi^2=\sum_s w_s\left(\hat D_{c,s}-\bar D\right)^2,\qquad p=P\!\left(\chi^2_{n-1}\ge\chi^2\right).$$

### 7.3 Dispersión extra
Con tantos puntos por eclipse, $\sigma_{\rm est}$ es tan pequeño que diferencias de 1–2 % ya son
"significativas". Para saber si la diferencia **importa**, se calcula la dispersión adicional
$\sigma_{\rm extra}$ que habría que sumar en cuadratura a cada error para que los sectores sean
consistentes ($\chi^2_{\rm red}=1$):

$$\sum_s\frac{(\hat D_{c,s}-\bar D)^2}{\sigma_{{\rm tot},s}^2+\sigma_{\rm extra}^2}=n-1 ,$$

y se expresa como fracción de la profundidad, $f_{\rm extra}=\sigma_{\rm extra}/\bar D$.

### 7.4 Marcas
| Marca | Condición | Interpretación |
|---|---|---|
| `consistente` | $p\ge0.0027$ | los sectores concuerdan dentro de sus errores |
| `difiere_leve` | $p<0.0027$ y $f_{\rm extra}<5\,\%$ | diferencia real pero pequeña: dilución o suavizado por cadencia |
| `difiere_revisar` | $p<0.0027$ y $f_{\rm extra}\ge5\,\%$ | **revisar**: diferencia grande entre sectores |
| `un_sector` | un solo sector válido | no hay con qué comparar |

### 7.5 Causas posibles de una diferencia
- **Dilución distinta por sector:** cada sector usa una apertura distinta, y en un campo tan poblado
  como Orión puede entrar más o menos luz de estrellas vecinas. Una contaminación $c$ reduce la
  profundidad a $D(1-c)$ y afecta **igual** al primario y al secundario, así que el cociente $D_2/D_1$ se
  conserva. Por eso la figura también muestra ese cociente.
- **Cadencia:** con 30 min (`texp_largo`) el mínimo se suaviza y la profundidad observada baja.
- **Manchas:** en primarias activas, una mancha ocultada durante el eclipse cambia la profundidad de un
  ciclo a otro.
- **Detrending o efeméride incorrectos:** ya corregidos en este notebook (§2.8, §2.9, §3), pero conviene
  revisar la figura de control de la sección 5.
- **Cambio real de inclinación:** una tercera estrella puede hacer precesar la órbita. En ese caso las
  profundidades cambian de forma sistemática con el tiempo (S6 → S32 → S98) y $D_2/D_1$ también puede
  cambiar.

### 7.6 Causa probable
Si las profundidades difieren pero $D_2/D_1$ varía menos de 10 % entre sectores, la causa más
probable es la **dilución**: la contaminación reduce los dos eclipses por igual. Si $D_2/D_1$ también
cambia, la dilución no lo explica y hay que revisar las curvas de esos sectores (figura de control de
la sección 5). Si no hay secundario, no se puede distinguir.

**Consecuencia para el ajuste.** Con dilución, la profundidad *más profunda* es la menos contaminada.
En starry puede tratarse con un parámetro de luz adicional por sector, o usando el sector menos
diluido para fijar la geometría.

Una figura por TIC se guarda en `resultados_pipeline/profundidades_v2/TIC_<tic>_sectores.png`, y la
tabla completa en `resultados_pipeline/profundidades_v2/consistencia_sectores.csv`.

In [ ]:
from scipy.stats import chi2 as dist_chi2
from scipy.optimize import brentq
from matplotlib.lines import Line2D

P_UMBRAL, F_UMBRAL = 0.0027, 0.05
CAD_COLOR = {"30 min": "#2a78d6", "10 min": "#eb6834", "≤ 6.7 min": "#1baf7a"}


def cadencia(sector):
    return "30 min" if sector in (5, 6) else ("10 min" if sector in (32, 33) else "≤ 6.7 min")


def consistencia(D, sD, alfa):
    # D, sD, alfa: arreglos por sector. Devuelve dict con la prueba de §7.
    D, sD, alfa = map(np.asarray, (D, sD, alfa))
    ok = np.isfinite(D) & np.isfinite(sD) & (sD > 0) & np.isfinite(alfa)
    D, sD, alfa = D[ok], sD[ok], alfa[ok]
    Dc = D * (1 + alfa / 4)
    st = np.hypot(sD, (alfa * D / 2) / np.sqrt(12))
    n = len(Dc)
    if n < 2:
        return dict(n=n, Dc=Dc, st=st, media=Dc.mean() if n else np.nan, media_err=st.mean() if n else np.nan,
                    chi2red=np.nan, p=np.nan, f_extra=np.nan, marca="un_sector")
    w = 1 / st ** 2
    m = np.sum(w * Dc) / w.sum()
    chi2 = np.sum(w * (Dc - m) ** 2)
    p = float(dist_chi2.sf(chi2, n - 1))

    def g(s):
        ww = 1 / (st ** 2 + s ** 2)
        mm = np.sum(ww * Dc) / ww.sum()
        return np.sum(ww * (Dc - mm) ** 2) - (n - 1)

    s_extra = 0.0 if g(0.0) <= 0 else brentq(g, 0.0, 10 * np.ptp(Dc) + 1e-12)
    f_extra = s_extra / m
    if p >= P_UMBRAL:
        marca = "consistente"
    elif f_extra < F_UMBRAL:
        marca = "difiere_leve"
    else:
        marca = "difiere_revisar"
    return dict(n=n, Dc=Dc, st=st, media=m, media_err=w.sum() ** -0.5, chi2red=chi2 / (n - 1), p=p,
                f_extra=f_extra, marca=marca)


# índice interno (si ya se corrió explore_priors_v2)
csv_idx = RUTA_ORION / "figuras_explore_priors" / "indice_sistemas.csv"
INDICE = dict(zip(*pd.read_csv(csv_idx)[["TIC", "indice"]].values.T)) if csv_idx.exists() else {}

filas_cons = []
for tic, g in med.groupby("TIC"):
    g = g.loc[~g["eclipse_ancho"]].sort_values("Sector")
    if g.empty:
        continue
    r1 = consistencia(g["D_primario"], g["D_primario_err"], g["alfa_primario"])
    g2 = g.loc[g["secundario_detectado"].astype(bool)]
    r2 = consistencia(g2["D_secundario"], g2["D_secundario_err"], g2["alfa_secundario"])
    cociente = (g2["D_secundario"] / g2["D_primario"]).values
    filas_cons.append(dict(
        TIC=tic, indice=INDICE.get(tic, np.nan), sectores=",".join(str(int(s)) for s in g["Sector"]),
        D1_media=r1["media"], D1_media_err=r1["media_err"], D1_chi2red=r1["chi2red"], D1_p=r1["p"],
        D1_f_extra=r1["f_extra"], marca_primario=r1["marca"],
        D2_media=r2["media"], D2_chi2red=r2["chi2red"], D2_p=r2["p"], D2_f_extra=r2["f_extra"],
        marca_secundario=r2["marca"] if r2["n"] else "sin_secundario",
        D2_D1_rango=(np.ptp(cociente) / np.mean(cociente)) if len(cociente) > 1 else np.nan,
    ))
cons = pd.DataFrame(filas_cons)


def causa(r):
    # §7.6: si las dos profundidades cambian pero su cociente no, la causa más probable es dilución
    if r["marca_primario"] not in ("difiere_leve", "difiere_revisar"):
        return ""
    if not np.isfinite(r["D2_D1_rango"]):
        return "sin secundario: no se puede distinguir"
    return "probable dilución (D2/D1 estable)" if r["D2_D1_rango"] < 0.10 else "no es dilución: revisar curvas"


cons["causa_probable"] = cons.apply(causa, axis=1)
cons = cons.sort_values(["marca_primario", "D1_f_extra"], ascending=[True, False])
cons.to_csv(DIR_FIG / "consistencia_sectores.csv", index=False)
print(cons["marca_primario"].value_counts().to_string())
cons.round(4)

In [ ]:
def figura_sectores(tic, ax=None, guardar=True):
    g = med.loc[(med["TIC"] == tic) & ~med["eclipse_ancho"]].sort_values("Sector")
    fila = cons.loc[cons["TIC"] == tic].iloc[0]
    propia = ax is None
    if propia:
        fig, ax = plt.subplots(figsize=(5.4, 4.3))
    x = np.arange(len(g))
    for nombre, col_D, col_e, col_a, mk, desplaz in (("primario", "D_primario", "D_primario_err", "alfa_primario", "o", -0.08),
                                                     ("secundario", "D_secundario", "D_secundario_err", "alfa_secundario", "s", 0.08)):
        sel = np.ones(len(g), bool) if nombre == "primario" else g["secundario_detectado"].astype(bool).values
        if not sel.any():
            continue
        r = consistencia(g[col_D].values[sel], g[col_e].values[sel], g[col_a].values[sel])
        xs = x[sel] + desplaz
        colores = [CAD_COLOR[cadencia(s)] for s in g["Sector"].values[sel]]
        # banda del promedio pesado
        if r["n"] >= 2:
            ax.axhspan(r["media"] - r["media_err"], r["media"] + r["media_err"], color="0.85", zorder=0)
            ax.axhline(r["media"], color="0.55", lw=1, zorder=1)
        ax.errorbar(xs, r["Dc"], yerr=r["st"], fmt="none", ecolor="0.35", elinewidth=1.2, capsize=3, zorder=2)
        ax.scatter(xs, r["Dc"], c=colores, marker=mk, s=55 if nombre == "primario" else 45,
                   edgecolor="0.2", lw=0.8, zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels([f"S{int(s)}" + ("*" if t > 0.2 else "") for s, t in zip(g["Sector"], g["texp_sobre_duracion"])],
                       fontsize=9)
    ax.set_xlim(-0.6, len(g) - 0.4)
    ax.set_ylabel("profundidad (corregida por sesgo)")
    idx = f"sistema {int(fila['indice'])} · " if np.isfinite(fila["indice"]) else ""
    marca = fila["marca_primario"]
    color_marca = {"consistente": "#1b7f3a", "difiere_leve": "#b07d00", "difiere_revisar": "#c0392b"}.get(marca, "0.4")
    # encabezado de 3 líneas arriba del panel: nombre, marca y causa probable
    linea2 = (f"primario: {marca}" + (f"  ·  f_extra = {100 * fila['D1_f_extra']:.1f} %,  p = {fila['D1_p']:.1e}"
                                       if np.isfinite(fila["D1_p"]) else ""))
    linea3 = " · ".join(t for t in [
        f"D2/D1 varía {100 * fila['D2_D1_rango']:.0f} %" if np.isfinite(fila["D2_D1_rango"]) else "",
        fila["causa_probable"] if isinstance(fila["causa_probable"], str) else ""] if t)
    ax.text(0.0, 1.30, f"{idx}TIC {tic}", transform=ax.transAxes, fontsize=10, va="bottom", ha="left")
    ax.text(0.0, 1.15, linea2, transform=ax.transAxes, fontsize=7.8, va="bottom", ha="left", color=color_marca,
            fontweight="bold" if marca == "difiere_revisar" else "normal")
    ax.text(0.0, 1.02, linea3, transform=ax.transAxes, fontsize=7.8, va="bottom", ha="left", color="0.35",
            style="italic")
    ax.margins(y=0.15)
    ax.grid(axis="y", alpha=0.25)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    if propia:
        ax.legend(handles=[Line2D([], [], ls="", marker="o", ms=7, mfc="0.6", mec="0.2", label="primario"),
                           Line2D([], [], ls="", marker="s", ms=6, mfc="0.6", mec="0.2", label="secundario")]
                  + [Line2D([], [], ls="", marker="o", ms=7, color=c, label=k) for k, c in CAD_COLOR.items()],
                  fontsize=7, loc="upper right", frameon=False, ncol=2)
        fig.tight_layout()
        if guardar:
            fig.savefig(DIR_FIG / f"TIC_{tic}_sectores.png", dpi=170, bbox_inches="tight")
        plt.close(fig)


# una figura por TIC (archivos) ...
for tic in cons["TIC"]:
    figura_sectores(tic)

# ... y una vista de conjunto en el notebook, ordenada por índice de sistema
orden = cons.sort_values("indice")["TIC"].tolist()
ncol = 4
nfil = int(np.ceil(len(orden) / ncol))
fig, axs = plt.subplots(nfil, ncol, figsize=(4.8 * ncol, 4.0 * nfil), squeeze=False,
                        gridspec_kw={"hspace": 0.75, "wspace": 0.35})
for ax, tic in zip(axs.flat, orden):
    figura_sectores(tic, ax=ax, guardar=False)
for ax in axs.flat[len(orden):]:
    ax.axis("off")
fig.legend(handles=[Line2D([], [], ls="", marker="o", ms=8, mfc="0.6", mec="0.2", label="primario"),
                    Line2D([], [], ls="", marker="s", ms=7, mfc="0.6", mec="0.2", label="secundario")]
           + [Line2D([], [], ls="", marker="o", ms=8, color=c, label=f"cadencia {k}") for k, c in CAD_COLOR.items()]
           + [Line2D([], [], color="0.55", lw=6, alpha=0.4, label="promedio pesado ± error")],
           loc="lower center", ncol=6, fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.01))
fig.suptitle("Profundidad de eclipse por sector (* = cadencia larga frente a la duración)", y=0.93)
fig.savefig(DIR_FIG / "consistencia_sectores_todos.png", dpi=130, bbox_inches="tight")
plt.show()

## 8) Curvas plegadas por sector: comparación visual de las profundidades

Complementa la sección 7. Para cada TIC se superponen las curvas de todos sus sectores válidos, en dos
paneles: el eclipse **primario** y el **secundario**.

- **Normalización:** la misma de las mediciones (§2.1). La tendencia biweight se calcula solo fuera de
  los eclipses y se divide; así cada sector queda en 1 fuera de eclipse y las profundidades se comparan
  directamente.
- **Plegado:** cada sector con **su propia efeméride** ($P_{\rm orb}$ y $t_0^{v2}$). El primario se centra
  en el mínimo medido ($c_1$) y el secundario en su fase medida ($c_2$), así que una órbita excéntrica
  no desplaza el panel.
- **Eje x:** horas desde el centro del eclipse, $\Delta t=(\phi-c)\,P_{\rm orb}\cdot24$. En horas, la
  duración también se compara directamente: si un sector se ve más ancho con la misma profundidad, es
  el suavizado por la cadencia (las curvas de 30 min tienen los bordes más redondeados).
- **Trazos:** puntos tenues (todos los datos) y la **mediana por bines** como línea gruesa. El ancho del
  bin es $\max(\tau/30,\ 1.2\,t_{\rm cad})$: nunca más fino que la cadencia del sector, para que las curvas
  de 30 min también tengan su línea. La ventana del secundario es 1.5 veces más ancha que la del primario,
  porque en órbitas excéntricas puede durar más. Las líneas discontinuas horizontales marcan $1-\hat D$ de cada sector, con el estimador de
  §2.3 sin corregir por sesgo; las profundidades corregidas por sesgo de la sección 7 serían algo
  mayores.
- **Encabezado:** la marca y la causa probable de la sección 7.

**Cómo leerla:**
- curvas superpuestas en primario y secundario: profundidad consistente;
- las dos curvas de un sector más someras **en la misma proporción** en ambos eclipses: dilución;
- un solo eclipse cambia, o la forma cambia: manchas, sistemáticos de ese sector o un cambio real
  de geometría.

Se guarda una figura por TIC en `resultados_pipeline/profundidades_v2/TIC_<tic>_plegadas.png`.

In [ ]:
COLOR_SECTOR = {5: "#2a78d6", 6: "#2a78d6", 32: "#eb6834", 33: "#eb6834", 87: "#eda100", 98: "#1baf7a"}


def curva_plegada(r):
    # flujo normalizado sin eclipses y fases centradas en el primario y en el secundario (horas)
    t, f = curvas[(r["TIC"], r["Sector"])]
    phi2 = r["fase_secundario"] if r["secundario_detectado"] else 0.5
    F, _ = normalizar_sin_eclipses(t, f, r["t0"], r["P"], r["W"], phi2)
    ph = fase(t, r["t0"], r["P"])
    c1 = r["centro_primario"] if np.isfinite(r["centro_primario"]) else 0.0
    c2 = r["fase_secundario"] if np.isfinite(r["fase_secundario"]) else 0.5
    h1 = (((ph - c1) + 0.5) % 1 - 0.5) * r["P"] * 24
    h2 = (((ph - c2) + 0.5) % 1 - 0.5) * r["P"] * 24
    return F, h1, h2


def mediana_bines(x, y, ancho, rango):
    bordes = np.arange(-rango, rango + ancho, ancho)
    xc, yc = [], []
    for a, b in zip(bordes[:-1], bordes[1:]):
        m = (x >= a) & (x < b)
        if m.sum() >= 2:
            xc.append(0.5 * (a + b)); yc.append(np.nanmedian(y[m]))
    return np.array(xc), np.array(yc)


def figura_plegadas(tic, axs=None, guardar=True):
    g = med.loc[(med["TIC"] == tic) & ~med["eclipse_ancho"]].sort_values("Sector")
    fila = cons.loc[cons["TIC"] == tic]
    fila = fila.iloc[0] if len(fila) else None
    propia = axs is None
    if propia:
        fig, axs = plt.subplots(1, 2, figsize=(11, 4.2), sharey=False)
    dur_h = g["Duration"].median() * 24
    rangos = (0.85 * dur_h, 1.25 * dur_h)   # el secundario puede durar más en órbitas excéntricas
    for _, r in g.iterrows():
        F, h1, h2 = curva_plegada(r)
        col = COLOR_SECTOR.get(int(r["Sector"]), "0.4")
        etiqueta = f"S{int(r['Sector'])}" + ("*" if r["texp_sobre_duracion"] > 0.2 else "")
        t_s, _ = curvas[(r["TIC"], r["Sector"])]
        cad_h = np.median(np.diff(t_s)) * 24
        ancho = max(dur_h / 30, 1.2 * cad_h)   # bin adaptativo: nunca más fino que la cadencia
        for ax, h, D, mostrar, rango in ((axs[0], h1, r["D_primario"], True, rangos[0]),
                                         (axs[1], h2, r["D_secundario"], bool(r["secundario_detectado"]), rangos[1])):
            if not mostrar:
                continue
            sel = np.abs(h) < rango
            ax.plot(h[sel], F[sel], ".", ms=1.8, color=col, alpha=0.22, zorder=1)
            xb, yb = mediana_bines(h, F, ancho, rango)
            ax.plot(xb, yb, "-", color=col, lw=2.2, zorder=3, label=etiqueta)
            if np.isfinite(D):
                ax.axhline(1 - D, color=col, lw=1, ls="--", alpha=0.9, zorder=2)
    for ax, nombre, rango in zip(axs, ("primario", "secundario"), rangos):
        ax.set_xlim(-rango, rango)
        ax.axhline(1, color="0.6", lw=0.8, zorder=0)
        ax.set_xlabel("fase")
        ax.set_title(f"eclipse {nombre}", fontsize=9, loc="right", color="0.35")
        ax.grid(alpha=0.2)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    axs[0].set_ylabel("flujo normalizado")
    if not g["secundario_detectado"].astype(bool).any():
        axs[1].text(0.5, 0.5, "secundario no detectado", transform=axs[1].transAxes, ha="center",
                    va="center", color="0.5", fontsize=9)
    # zoom vertical en el secundario para que se vea su diferencia (es más somero)
    sec = g.loc[g["secundario_detectado"].astype(bool), "D_secundario"]
    if len(sec):
        dmax = np.nanmax(sec)
        axs[1].set_ylim(1 - 1.35 * dmax - 0.004, 1 + 0.25 * dmax + 0.004)
    dmax1 = np.nanmax(g["D_primario"])
    axs[0].set_ylim(1 - 1.25 * dmax1 - 0.004, 1 + 0.2 * dmax1 + 0.004)
    axs[0].legend(fontsize=8, loc="lower left", frameon=False, title="sector", title_fontsize=8)
    idx = INDICE.get(tic, np.nan)
    nombre = (f"sistema {int(idx)} · " if np.isfinite(idx) else "") + f"TIC {tic}"
    if fila is not None:
        color_marca = {"consistente": "#1b7f3a", "difiere_leve": "#b07d00",
                       "difiere_revisar": "#c0392b"}.get(fila["marca_primario"], "0.4")
        sub = fila["marca_primario"] + (f"  ·  {fila['causa_probable']}" if isinstance(fila["causa_probable"], str)
                                        and fila["causa_probable"] else "")
    else:
        color_marca, sub = "0.4", ""
    axs[0].text(0.0, 1.13, nombre, transform=axs[0].transAxes, fontsize=10, va="bottom")
    axs[0].text(0.0, 1.03, sub, transform=axs[0].transAxes, fontsize=8, va="bottom", color=color_marca,
                fontweight="bold" if fila is not None and fila["marca_primario"] == "difiere_revisar" else "normal")
    if propia:
        fig.tight_layout()
        if guardar:
            fig.savefig(DIR_FIG / f"TIC_{tic}_plegadas.png", dpi=170, bbox_inches="tight")
        plt.close(fig)


with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for tic in cons["TIC"]:
        figura_plegadas(tic)

    # vista de conjunto: dos sistemas por fila (primario | secundario), ordenados por índice
    orden = cons.sort_values("indice")["TIC"].tolist()
    nfil = int(np.ceil(len(orden) / 2))
    fig, axs = plt.subplots(nfil, 4, figsize=(20, 3.9 * nfil),
                            gridspec_kw={"hspace": 0.85, "wspace": 0.28})
    for k, tic in enumerate(orden):
        fila_ax = axs[k // 2]
        figura_plegadas(tic, axs=fila_ax[2 * (k % 2): 2 * (k % 2) + 2], guardar=False)
    for k in range(len(orden), 2 * nfil):
        for ax in axs[k // 2][2 * (k % 2): 2 * (k % 2) + 2]:
            ax.axis("off")
    fig.suptitle("Eclipses plegados por sector (* = cadencia larga frente a la duración; "
                 "línea discontinua = 1 − D de cada sector)", y=0.995, fontsize=13)
    fig.savefig(DIR_FIG / "plegadas_sectores_todos.png", dpi=110, bbox_inches="tight")
    plt.show()